##Project Overview

###Objective
**Second Brain** is designed to help you consume the large documents (papers, financal reports, market reports, etc.) and increase your productticity. It leverage the LLM to consume the documents in Google Drives Folders and answer the inqueries.

Smart Assistant
is to create a second brain which can help us consume the large documents and increase the productivity.

###Challenges

* Many papers and reports are PDF format, it is not easy to transform them to a structure data and search the answers.
* The answser may not comes from a single file but the combination of the multiple files

###Recognition
This works is inspired by [How to build a ChatGPT + Google Drive app with LangChain and Python](https://www.haihai.ai/gpt-gdrive/?ref=emergentmind)

### Wroking Environment and Preparation

* Copy this notebook to your account
* Make sure the Vertex AI API are enabled and billable
* You can use the exsiting [Google Folder (with files downloaded)](https://drive.google.com/drive/folders/1RZvvar4lDuT6w-kaDxr8hkHFTuI4np_C), the ID already assgined in this sample code. Or create your own foler to store the documents. Following are PDF files need to use for this demo.
  * Alphabet Annual Report: [2021](https://www.abc.xyz/assets/9a/bd/838c917c4b4ab21f94e84c3c2c65/goog-10-k-q4-2022.pdf), [2022](https://www.abc.xyz/assets/d9/85/b7649a9f48c4960adbce5bd9fb54/20220202-alphabet-10k.pdf)
  * Amazon Annual Report: [2021](https://s2.q4cdn.com/299287126/files/doc_financials/2022/ar/Amazon-2021-Annual-Report.pdf), [2022](https://s2.q4cdn.com/299287126/files/doc_financials/2023/ar/Amazon-2022-Annual-Report.pdf)
* Assigned the variable of the demo environment in follow session.

Identify the Project ID for Vertex AI API, Location, and Google Drive Folder ID.

In [ ]:
# Replace these to your own

PROJECT_ID = ""  # @param {type:"string"}
LOCATION = "us-central1" # @param {type:"string"}
FOLDER_ID = "" # @param {type:"string"}

In [ ]:
from platform import python_version
print(python_version())

3.10.6


In [ ]:
# Install Vertex AI LLM SDK, langchain and dependencies
# Remember to click RESTART RUNTIME after this step.
! pip install config --upgrade --user
! pip install google-cloud-aiplatform google-api-python-client

In [ ]:
# Install langchain and dependencies
! pip install langchain chromadb pypdf2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.0/399.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 4.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 16.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 29.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 34.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.3 MB/s 

### Authenticating your notebook environment

It will pop up the authorization window for the Google Drive access.

In [ ]:
from google.colab import auth as google_auth
google_auth.authenticate_user()

### Import libraries

In [ ]:
import langchain
import vertexai

# Vertex AI
from google.cloud import aiplatform
from langchain.llms import VertexAI

vertexai.init(project=PROJECT_ID, location=LOCATION)

print(f"LangChain version: {langchain.__version__}")
print(f"Vertex AI SDK version: {aiplatform.__version__}")

LangChain version: 0.0.243
Vertex AI SDK version: 1.28.1


Load the documents and store into the Chroma Database

In [ ]:
from langchain.document_loaders import GoogleDriveLoader

loader = GoogleDriveLoader(
    folder_id=FOLDER_ID,
    recursive=False
)
docs = loader.load()

DependencyError: ignored

In [ ]:
# split the documents into chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=200)
docs = text_splitter.split_documents(docs)

print(f"# of chunks = {len(docs)}")

In [ ]:
from langchain.embeddings import VertexAIEmbeddings
from langchain.vectorstores import Chroma

# Embedding
embeddings = VertexAIEmbeddings()
db = Chroma.from_documents(docs, embeddings)

In [ ]:
# Create chain to answer questions
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# LLM model
llm = VertexAI(
    model_name="text-bison@001",
    max_output_tokens=256,
    temperature=0,
    top_p=0.95,
    top_k=40,
    verbose=True,
)

prompt_template = """
You are a trustworthy consultant.
Answer the users question based on the context only.
Do not make up data. If you cannot find answer in the context, then say 'Sorry, I cannot find the answers in the documents.'

{context}

Question: {question}
"""

PROMPT = PromptTemplate(
    template = prompt_template, input_variables=["context", "question"]
)

chain_type_kwargs = {"prompt": PROMPT}

# Seting the approriate retriver
#retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever = db.as_retriever(search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.4, "k": 4})

qa = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs=chain_type_kwargs,
)

###Asking Questions###

#### Asking questions which can be found in the single file.
Alphabet Net Income
* 2022: \$59,972 millions (in 2022 report)
* 2021: \$76,033 millions (in 2022, 2021 report)
* 2020: \$40,269 millions (in 2022, 2021 report)
* 2019: \$34,343 millions (in 2021 report)

In [ ]:
query = "What is the net income of Alphabet in 2022?"
result = qa({"query": query})

print(f"Query: ", result["query"])
print(f"Answer: ", result["result"])
print(f"Source: ", result["source_documents"])

Query:  What is the net income of Alphabet in 2022?
Answer:  The net income of Alphabet in 2022 was $59,972 million.
Source:  [Document(page_content='Alphabet Inc.\nCONSOLIDATED STATEMENTS OF INCOME\n(in millions, except per share amounts)\n Year Ended December 31,\n 2020 2021 2022\nRevenues $ 182,527 $ 257,637 $ 282,836 \nCosts and expenses:\nCost of revenues  84,732  110,939  126,203 \nResearch and development  27,573  31,562  39,500 \nSales and marketing  17,946  22,912  26,567 \nGeneral and administrative  11,052  13,510  15,724 \nTotal costs and expenses  141,303  178,923  207,994 \nIncome from operations  41,224  78,714  74,842 \nOther income (expense), net  6,858  12,020  (3,514) \nIncome before income taxes  48,082  90,734  71,328 \nProvision for income taxes  7,813  14,701  11,356 \nNet income $ 40,269 $ 76,033 $ 59,972 \nBasic net income per share of Class A, Class B, and Class C stock $ 2.96 $ 5.69 $ 4.59 \nDiluted net income per share of Class A, Class B, and Class C stock 

#### Asking questions which can be found in the multiple files.

Alphabet Net Income
* 2022: \$59,972 millions (in 2022 report)
* 2021: \$76,033 millions (in 2022, 2021 report)
* 2020: \$40,269 millions (in 2022, 2021 report)
* 2019: \$34,343 millions (in 2021 report)

In [ ]:
query = "List the net income of Alphabet in 2019 to 2022 in millions"
result = qa({"query": query})

print(f"Query: ", result["query"])
print(f"Answer: ", result["result"])
print(f"Source: ", result["source_documents"])

Query:  List the net income of Alphabet in 2019 to 2022 in millions
Answer:  The net income of Alphabet in 2019 was $34,343 million.
The net income of Alphabet in 2020 was $40,269 million.
The net income of Alphabet in 2021 was $76,033 million.
The net income of Alphabet in 2022 was $59,972 million.
Source:  [Document(page_content='Alphabet Inc.\nCONSOLIDATED STATEMENTS OF INCOME\n(In millions, except per share amounts)\n Year Ended December 31,\n 2019 2020 2021\nRevenues $ 161,857 $ 182,527 $ 257,637 \nCosts and expenses:\nCost of revenues  71,896  84,732  110,939 \nResearch and development  26,018  27,573  31,562 \nSales and marketing  18,464  17,946  22,912 \nGeneral and administrative  9,551  11,052  13,510 \nEuropean Commission fines  1,697  0  0 \nTotal costs and expenses  127,626  141,303  178,923 \nIncome from operations  34,231  41,224  78,714 \nOther income (expense), net  5,394  6,858  12,020 \nIncome before income taxes  39,625  48,082  90,734 \nProvision for income taxes  

Google Cloud Revenue:
* 2022: \$26,280 millions (in 2022 report)
* 2021: \$19,206 millions (in 2021 report)

In [ ]:
query = "What's the revenue of Google Cloud in 2021 and 2022?"
result = qa({"query": query})

print(f"Query: ", result["query"])
print(f"Answer: ", result["result"])
print(f"Source: ", result["source_documents"])

Query:  What's the revenue of Google Cloud in 2021 and 2022?
Answer:  Google Cloud's revenue was $19.2 billion in 2021 and $26.28 billion in 2022.
Source:  [Document(page_content='Note 2.    Revenues  \nDisaggregated Revenues\nThe following table presents revenues disaggregated by type (in millions):\nYear Ended December 31,\n2020 2021 2022\nGoogle Search & other $ 104,062 $ 148,951 $ 162,450 \nYouTube ads  19,772  28,845  29,243 \nGoogle Network  23,090  31,701  32,780 \nGoogle advertising  146,924  209,497  224,473 \nGoogle other  21,711  28,032  29,055 \nGoogle Services total  168,635  237,529  253,528 \nGoogle Cloud  13,059  19,206  26,280 \nOther Bets  657  753  1,068 \nHedging gains (losses)  176  149  1,960 \nTotal revenues $ 182,527 $ 257,637 $ 282,836 \nNo individual customer or groups of affiliated customers represented more than 10% of our revenues in 2020 , \n2021 , or 2022 .\nThe following table presents revenues disaggregated by geography, based on the addresses of our cu

Amazon Employee Number:
* Dec 31, 2022: 1,541,000 (in 2022 report)
* Dec 31, 2021: 1,608,000 (in 2021 report)

In [ ]:
query = "How many employee of Amazon in end of 2021 and 2022?"
result = qa({"query": query})

print(f"Query: ", result["query"])
print(f"Answer: ", result["result"])
print(f"Source: ", result["source_documents"])

Query:  How many employee of Amazon in end of 2021 and 2022?
Answer:  Amazon employed approximately 1,608,000 full-time and part-time employees as of December 31, 2021. As of December 31, 2022, Amazon employed approximately 1,541,000 full-time and part-time employees.
Source:  [Document(page_content='Seasonality\nOur business is affected by seasonality, which historically has resulted in higher sales volume during our fourth quarter, \nwhich ends December 31. \nHuman Capital\nOur employees are critical to our mission of being Earth’s most customer-centric company. As of December 31, 2021, we \nemployed approximately 1,608,000 full-time and part-time employees. Additionally, we use independent contractors and \ntemporary personnel to supplement our workforce. Competition for qualified personnel is intense, particularly for software \nengineers, computer scientists, and other technical staff, and constrained labor markets have increased competition for \npersonnel across other parts of o

#### Asking somthing which cannot find in the folder.

In [ ]:
query = "The distance between Sun and Earth?"
result = qa({"query": query})

print(f"Query: ", result["query"])
print(f"Answer: ", result["result"])
print(f"Source: ", result["source_documents"])

Query:  The distance between Sun and Earth?
Answer:  Sorry, I cannot find the answers in the documents.
Source:  []
